# Load libraries

In [ ]:
import re
import os
import random
import pandas as pd
import numpy as np

from tqdm.auto import tqdm

random.seed(101)

In [ ]:
data_loc = "/kaggle/input/pii-detect-toa-datasets/"

# Read TOA data

In [ ]:
toa_df = pd.DataFrame()

for file in tqdm(os.listdir(data_loc)):
    
    if '_ft' in file:
        continue
    
    path = data_loc + file
    temp = pd.read_csv(path)
    
    toa_df = pd.concat([toa_df, temp], ignore_index=True)

toa_df = toa_df[toa_df.index != 5318]
toa_df.drop_duplicates(inplace=True)
toa_df = toa_df.reset_index(drop=True)

toa_df.shape    

In [ ]:
toa_df.head()

# Clean up PIIs

In [ ]:
def clean_piis(piis_column):
    # Define the regex patterns for the different conditions
    patterns_to_remove = [
        r'ID_NUM: <insert-generated-id>\n?',  # Matches 'ID_NUM' with '<insert-generated-id>'
        r'STREET_ADDRESS: <insert-generated-street-address>\n?',  # Matches 'STREET_ADDRESS' with '<insert-generated-street-address>'
        r'EMAIL: \[insert-generated-email\]\n?',
        r'URL_PERSONAL: \[insert-generated-url\]\n?',
        r'URL_PERSONAL: \[InsertGeneratedUrl\]\n?',
        r'Generated\n?',
        '\* Generated\\_\n',
        '```\n',
        r'\[insert-generated-url\]\(?',
        'This generated\n',
        'EMAIL: .*[Gg]enerate.*\n',
        'ID_NUM: .*[Gg]enerate.*\n',
        'NAME_STUDENT: .*[Gg]enerate.*\n',
        'PHONE_NUM: .*[Gg]enerate.*\n',
        'STREET_ADDRESS: .*[Gg]enerate.*\n',
        'URL_PERSONAL: .*[Gg]enerate.*\n',
        'USERNAME: .*[Gg]enerate.*\n',
        
    ]
    
    pattern_to_replace = {
        r'Generated\\_addr:': 'STREET_ADDRESS:',  # Replaces 'Generated\_addr' with 'STREET_ADDRESS'
        r'\\_addr': 'STREET_ADDRESS:',
    }
    
    # Apply the removal patterns
    for pattern in patterns_to_remove:
        piis_column = piis_column.apply(lambda x: re.sub(pattern, '', x))
    
    # Apply the replacement patterns
    for key, value in pattern_to_replace.items():
        piis_column = piis_column.apply(lambda x: re.sub(key, value, x))
    
    return piis_column



In [ ]:
toa_df['piis'] = toa_df['piis'].astype(str)
toa_df['piis'] = clean_piis(toa_df['piis'])

In [ ]:
temp = toa_df[toa_df.piis.notnull()].copy()

print(temp[temp['piis'].str.contains("enerate")].shape)

# temp[temp['piis'].str.contains("enerate")].piis.sample().values[0]

# Clean up Essays

In [ ]:
print(toa_df['essays'].sample().values[0])

In [ ]:
def replace_section_headings_randomly(essay):
    # Define all possible replacement formats for each section
    replacement_formats = [
        "{number}. {section_title}\n",  # Numbered sections
        "{section_title}:\n",  # Section title followed by a colon
        "{section_title}\n",  # Just the section title
        
        "{number}. {section_title} ",
        "{section_title}: ",
        "{section_title} ",
        
        ""
    ]
    
    caps = [1,0,-1]
    
    # Randomly select a format for this essay
    selected_format = random.choice(replacement_formats)
    case_format = random.choice(caps)
    
    # Define the sections to replace and their 'proper' titles
    sections = [
        ("CHALLENGES", "Challenge"),
        ("SELECTION", "Selection"),
        ("INSIGHT", "Insight"),
        ("APPLICATION", "Application"),
        ("APPROACH", "Approach"),
    ]
    
    # Process each section
    for original, title in sections:
        # Generate a regex pattern to match all case variations of the section
        pattern = r'\[?(' + '|'.join([original, original.capitalize(), original.lower()]) + r's?)\]?\s*'
        
        # Determine the replacement string based on the selected format
        if "{number}" in selected_format:
            # Extract the number for the current section
            section_number = sections.index((original, title)) + 1
            replacement = selected_format.format(number=section_number, section_title=title)
        elif selected_format == "":
            replacement = ""
        else:
            replacement = selected_format.format(section_title=title)
        
        if case_format==1:
            replacement = replacement.upper()
        elif case_format==0:
            replacement = replacement.lower()
        
        # Replace the section header in the essay
        essay = re.sub(pattern, replacement, essay, flags=re.IGNORECASE)
    
    return essay

In [ ]:
%%time

toa_df['essays'] = toa_df['essays'].apply(replace_section_headings_randomly)

In [ ]:
print(toa_df['essays'].sample().values[0])

# Check PIIs in Essay

In [ ]:
# Define a function to identify included and missing PIIs in each essay
def identify_piis(row):
    if row['include'] and pd.notnull(row['piis']):
        
        try:
            # Extract the specified PIIs from the 'piis' column
            specified_piis = [pii.split(": ")[1].strip() for pii in row['piis'].split("\n") if ': ' in pii]
        except:
            print(row)
        included_piis = []
        missing_piis = []
        
        # Check each specified PII for its presence in the essay
        for pii in specified_piis:
            if pii in row['essays']:
                included_piis.append(pii)
            else:
                missing_piis.append(pii)
                
        return included_piis, missing_piis
    else:
        # If the essay should not include PIIs or no PIIs are specified, return empty lists
        return [], []

# Apply the function to each row in the dataframe and create new columns for included and missing PIIs
toa_df[['PII_Included', 'PII_Missing']] = toa_df.apply(identify_piis, axis=1, result_type='expand')

toa_df['Count_PII_Included'] = toa_df['PII_Included'].apply(len)
toa_df['Count_PII_Missing'] = toa_df['PII_Missing'].apply(len)

# Display the modified dataframe to verify the changes
toa_df.head()

In [ ]:
toa_df['Count_PII_Included'].value_counts()

In [ ]:
toa_df['piis'].values[109]

In [ ]:
toa_df.dtypes

In [ ]:
toa_df[toa_df.index==3071]

In [ ]:
toa_df[toa_df.index==5318]['piis'].values[0]

# Replace repeated PIIs

In [ ]:
def update_phone_numbers_to_pattern(pii_dict):

    # Define the pattern
    pattern = r'\(\d{3}\)\d{3}-\d{4}'
    
    
    if pii_dict == {}:
        return {}
    
    # Highest key number for naming new keys
    highest_key_num = max([int(re.search(r'\d+', key).group()) for key in pii_dict.keys()])
    pii_dict_copy = pii_dict.copy()
    for key, val in pii_dict_copy.items():
        if not re.match(pattern, val) and 'PHONE_NUM' in key:
            # Extract digits from the phone number
            digits = re.sub(r'\D', '', val)
            
            # Ensure there are at least 10 digits to form a phone number, truncate or pad if necessary
            clean_digits = (digits[:10] if len(digits) >= 10 else digits)[:10]
            
            # Transform into the specified pattern
            new_phone = f'({clean_digits[:3]}){clean_digits[3:6]}-{clean_digits[6:]}'
            new_key_num = highest_key_num + 1
            new_key = f'PHONE_NUM_{new_key_num}'
            pii_dict[new_key] = new_phone
            
            highest_key_num += 1  # Increment for the next key if needed
    
    return pii_dict

In [ ]:
def replace_pii_patterns(text):
    
    text = text.replace(f"{pii_id}{pii_id}", f"{pii_id}")
    text = text.replace(f"{pii_id} {pii_id}", f"{pii_id}")
    text = text.replace(f"{pii_id}:{pii_id}", f"{pii_id}")
    text = text.replace(f"{pii_id}: {pii_id}", f"{pii_id}")
    text = text.replace(f"{pii_id} : {pii_id}", f"{pii_id}")
    
    text = text.replace(f"[{pii_id}]({pii_id})", f"{pii_id}")
    text = text.replace(f"{pii_id}]({pii_id}", f"{pii_id}")
    
    return text

In [ ]:
# index=47
# row = toa_df.loc[index]

# Iterate over each row in the dataframe `toa_df`, using tqdm for progress visualization
for index, row in tqdm(enumerate(toa_df.itertuples()), total=len(toa_df)):
    
    # Replace carriage return and newline characters with newline in the essay text
#     full_text = getattr(row, 'essays').replace('\r\n', '\n')
    full_text = getattr(row, 'essays')

    # Check if the row has any PII items
    if getattr(row, 'numpii') > 0:

        # Construct a dictionary of PII items from the row, ensuring they are properly formatted
        pii_dict = {f"{pii.split(': ')[0].strip()}_{i}": pii.split(": ")[1].strip() for i, pii in enumerate(getattr(row, 'piis').split("\n")) if len(pii) > 10 and ': ' in pii}

        pii_dict = update_phone_numbers_to_pattern(pii_dict)

        # Iterate over each PII item to label tokens accordingly
        for pii_type, pii_id in pii_dict.items():

            # Clean pii_type by removing colons
            pii_type = pii_type.replace(":", "")

            full_text = replace_pii_patterns(full_text)

        toa_df.loc[index, 'essays'] = full_text


# Convert to Competition format

In [ ]:
from spacy.lang.en import English
en_tokenizer = English().tokenizer

def tokenize_with_spacy(text, tokenizer=en_tokenizer):
    tokenized_text = tokenizer(text)
    tokens = [token.text for token in tokenized_text]
    trailing_whitespace = [bool(token.whitespace_) for token in tokenized_text]
    return tokens, trailing_whitespace

In [ ]:
def replace_phone_number_format(text):
    # This pattern matches phone numbers in the format (xxx) xxx-xxxx
    pattern = r'\(\d{3}\)\s\d{3}-\d{4}'
    # This replacement pattern removes the space after the area code
    replacement = lambda match: match.group(0).replace(' ', '')
    # Substitute the found patterns in the text with the new format
    new_text = re.sub(pattern, replacement, text)
    return new_text

In [ ]:
# Initialize an empty list to store processed data
toa_data = []

# Set starting document ID for identification purposes
start_document_id = 1000000

phone_number_pattern = r'\(\d{3}\)\d{3}-\d{4}'

# index = 3980
# row = toa_df.loc[index]

# if True:
# Iterate over each row in the dataframe `toa_df`, using tqdm for progress visualization
for index, row in tqdm(enumerate(toa_df.itertuples()), total=len(toa_df)):
    
    # Replace carriage return and newline characters with newline in the essay text
    full_text = getattr(row, 'essays').replace('\r\n', '\n')
    full_text = replace_phone_number_format(full_text)
    
    # Tokenize the full_text using a pre-defined Spacy tokenizer, capturing tokens and their trailing whitespace
    tokens, trailing_whitespace = tokenize_with_spacy(full_text, en_tokenizer)
    
    # Initialize labels for each token as 'O' (Outside), indicating no PII by default
    labels = ['O'] * len(tokens)
    
    # Check if the row has any PII items
    if getattr(row, 'numpii') > 0:
        
        # Construct a dictionary of PII items from the row, ensuring they are properly formatted
        pii_dict = {f"{pii.split(': ')[0].strip()}_{i}": pii.split(": ")[1].strip() for i, pii in enumerate(getattr(row, 'piis').split("\n")) if len(pii) > 10 and ': ' in pii}        
        pii_dict = update_phone_numbers_to_pattern(pii_dict)
        
        for pii_type, pii_id in pii_dict.items():
            
            pattern = r'\[([^\]]+)\]\(([^)]+)\)'
            pii_dict[pii_type] = re.sub(pattern, r'\2', pii_id)

        # Iterate over each PII item to label tokens accordingly
        for pii_type, pii_id in pii_dict.items():
            
            # Clean pii_type by removing colons
            pii_type = pii_type.replace(":", "")
            
            # Special handling for STREET_ADDRESS to consider each part separately
            if 'STREET_ADDRESS' in pii_type:
                temp, _ = tokenize_with_spacy(pii_id, en_tokenizer)
                pii_id_ = [x.strip(",") for x in temp]
            else:
                # pii_id_ = pii_id.split()
                pii_id_, _ = tokenize_with_spacy(pii_id, en_tokenizer)

            # Iterate over tokens to assign labels based on PII items
            for i, token in enumerate(tokens):
                
                if token in pii_id_:

                    # Mark the beginning of a PII entity
                    labels[i] = f"B-{pii_type[:-2]}" 

                    # If the previous token is part of the same PII entity, mark as continuation (I-)
                    if labels[i-1] == f"B-{pii_type[:-2]}":
                        labels[i] = f"I-{pii_type[:-2]}"
                    
                    # Special handling for STREET_ADDRESS to ensure continuous entities are labeled correctly
                    if f"{pii_type[:-2]}" in labels[i] and f"{pii_type[:-2]}" in labels[i-2]  and 'STREET_ADDRESS' in pii_type:
                        labels[i] = f"I-{pii_type[:-2]}"
                        labels[i-1] = f"I-{pii_type[:-2]}"
                
                # Special handling for phone numbers to ensure correct tokenization and labeling
                if 'PHONE_NUM' in pii_type and re.match(phone_number_pattern, pii_id):

                    tokens_ph, _ = tokenize_with_spacy(pii_id, en_tokenizer)

                    for token_ph in tokens_ph:
                        
                        if token == token_ph and (i+1) < len(tokens):
                            
                            labels[i] = f"B-PHONE_NUM"
                        
                        if token == token_ph:
                            
                            if 'PHONE_NUM' in labels[i-1]:
                                labels[i] = f"I-PHONE_NUM"
            
            # Post-processing to correct labels for standalone or incorrect B-PHONE_NUM labels
            for i, token in enumerate(tokens):
                
                if i+1 < len(tokens):
                    if labels[i] == 'B-PHONE_NUM' and "PHONE_NUM" not in labels[i+1] and len(token) < 3:
                        labels[i] = "O"
                
                if i+2 < len(tokens):
                    if labels[i] == "O" and token.lower() == "apt" and tokens[i+1]=="." and labels[i-1] == "I-STREET_ADDRESS" and labels[i+2] == "B-STREET_ADDRESS":
                        
                        labels[i] = "I-STREET_ADDRESS"
                        labels[i+1] = "I-STREET_ADDRESS"
                        labels[i+2] = "I-STREET_ADDRESS"
    
    # Post-processing to correct labels for standalone or incorrect B-STREET_ADDRESS labels
    for i, label in enumerate(labels):
        
        # Correct labeling for the last token or isolated B-STREET_ADDRESS tokens
        if label == 'B-STREET_ADDRESS' and ((i+1 >= len(labels)) or (labels[i+1] == 'O')):
            labels[i] = 'O'
        
        if i+1 < len(tokens):
            if label == 'I-PHONE_NUM' and tokens[i] == '(' and labels[i+1] == 'O':
                labels[i] = 'O'
            
            if label == 'I-PHONE_NUM' and tokens[i] == '(' and tokens[i-1] == '(':
                labels[i-1] = 'O'
                labels[i] = 'B-PHONE_NUM'
                
    # Additional post-processing to handle specific STREET_ADDRESS labeling issues
    for i, (token, label) in enumerate(zip(tokens, labels)):
        
        # Correct labeling for specific pattern related to STREET_ADDRESS
        if i + 3 < len(tokens):
            if label == 'B-STREET_ADDRESS' and labels[i+1] == 'I-STREET_ADDRESS' and labels[i+2] == 'I-STREET_ADDRESS' and labels[i+3] == 'O' and tokens[i+1]=='to':
                labels[i] = 'O'
                labels[i+1] = 'O'
                labels[i+2] = 'O'
        
        if i+1 < len(tokens):
            
            if 'I-' in labels[i] and 'B-' in labels[i+1]:
                if labels[i].strip('I-') == labels[i+1].strip('B-'):
                    labels[i+1] = labels[i+1].replace('B-', 'I-') 
    
    
    for i in range(len(tokens)):
        
        # URL_PERSONAL cleanup
        if tokens[i] in ['(','[',')',']','-'] and labels[i]=='B-URL_PERSONAL':
            
            labels[i] = 'O'
            
            if i+1 < len(tokens):
            
                if labels[i+1] == 'I-URL_PERSONAL':
                    labels[i+1] = 'B-URL_PERSONAL'
        
        # STREET_ADDRESS cleanup
        if len(tokens[i])==1 and labels[i]=='B-STREET_ADDRESS':
            labels[i] = 'O'
        
                
        
    
    # Append processed data to toa_data list
    toa_data.append({
                    "document": start_document_id + index,
                    "full_text": full_text,
                    "tokens": tokens,
                    "trailing_whitespace": trailing_whitespace,
                    "labels": labels
                    })


In [ ]:
# pii_dict

In [ ]:
# row['PII_Missing']

In [ ]:
# tokens = toa_data[0]['tokens']
# labels = toa_data[0]['labels']

# for token, label in zip(tokens, labels):
#     print(f"{token}: {label}")

# Explore

In [ ]:
import spacy
# load the spacy pipline model
nlp = spacy.load("en_core_web_lg")

# spacy color options for manual render
options = {"colors": {"NAME_STUDENT": "#748CAB", 
                      "URL_PERSONAL": "#FFFC31", 
                      "ID_NUM": "#E94F37", 
                      "EMAIL": "#F8B195", 
                      "STREET_ADDRESS": "#BDBF09", 
                      "PHONE_NUM": "#D96C06", 
                      "USERNAME": "#2292A4"}}

In [ ]:
#Function to convert a single row from the dataframe to SpaCy format
def convert_to_spacy_format(text, tokens, labels, trailing_whitespace):
    ents = []  # To store entity dictionaries
    start = 0  # Position tracker for the start of each token in the text
    
    for i, (token, label, space) in enumerate(zip(tokens, labels, trailing_whitespace)):
        if label.startswith('B-') or label.startswith('I-'):
            label_type = label[2:]  # Extract entity type from label
            token_start = text.find(token, start)  # Find the start index of the token in text
            token_end = token_start + len(token)  # Calculate the end index of the token
            
            # If it's a 'B-' label or the first 'I-' label following non-matching or 'O' labels, start a new entity
            if label.startswith('B-') or (label.startswith('I-') and (i == 0 or not labels[i-1].endswith(label_type))):
                ents.append({"start": token_start, "end": token_end, "label": label_type})
            # If it's an 'I-' label continuing an entity, extend the last entity's end index
            elif label.startswith('I-') and ents and ents[-1]["label"] == label_type:
                ents[-1]["end"] = token_end
            
            start = token_end + (1 if space == 'True' else 0)  # Update start position for next token
        
        else:
            start = start + len(token)
    
    return [{"text": text, "ents": ents, "title": None}]
def encode_labels(df):
    df["unique_labels"] = df["labels"].apply(lambda x: list(set(
        [l.split('-')[1] for l in x if l != 'O']
         )))
    # add 1-hot encoding
    from sklearn.preprocessing import MultiLabelBinarizer

    mlb = MultiLabelBinarizer()
    one_hot_encoded = mlb.fit_transform(df['unique_labels'])
    one_hot_df = pd.DataFrame(one_hot_encoded, columns=mlb.classes_)
    df = pd.concat([df, one_hot_df], axis=1)
    
    # add 'OTHER' column
    df['OTHER'] = df['unique_labels'].apply(lambda x: 1 if len(x) == 0 else 0)
    
    return df, list(mlb.classes_) + ['OTHER']

In [ ]:
index=4868

row = toa_df.loc[index]

row

In [ ]:
row['PII_Included']

In [ ]:
# # 5318, 3071, 2952

# df = pd.DataFrame(toa_data)

# filtered_row = df.loc[index]

# # Assuming there's only one such row, getting the `full_text` content
# display_text = filtered_row['full_text'] if not filtered_row.empty else "Document not found"
# display_labels = filtered_row['labels'] if not filtered_row.empty else "No Labels found"
# trailing_whitespace = filtered_row['trailing_whitespace'] if not filtered_row.empty else []
# tokens = filtered_row['tokens'] if not filtered_row.empty else []

# # Cleanup new line for SpaCy rendering
# display_text = display_text.replace("\n\n", "\r\n")

# # Get labels from dataframe and convert to SpaCy format
# ex = convert_to_spacy_format(display_text, tokens, display_labels, trailing_whitespace)        

In [ ]:
# tokens = getattr(filtered_row, 'tokens')
# labels = getattr(filtered_row, 'labels')

# for token, label in zip(tokens, labels):
#     print(f"{token}: {label}")

In [ ]:
# Display labels from dataframe
# spacy.displacy.render(ex, style="ent", manual=True, jupyter=True, options=options)

# Save to json

In [ ]:
len(toa_data)

In [ ]:
# Path where the JSON file will be saved
file_path = './illi_data2.json'

# Saving the list of dictionaries to a JSON file
import json

with open(file_path, 'w') as file:
    json.dump(toa_data, file, indent=4)